<a href="https://colab.research.google.com/github/Adamphoenix003/GNN-LinkPrediction/blob/main/CoraDatasetfinal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install node2vec


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 38.0 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is inc

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)
from node2vec import Node2Vec
from sklearn.linear_model import LogisticRegression
import random

In [ ]:


content_path = "/content/sample_data/cora.content"

cora_content = pd.read_csv(
    content_path,
    sep="\t",
    header=None
)

print("Shape:", cora_content.shape)
cora_content.head()


Shape: (2708, 1435)


,0,1,2,3,4,5,6,7,8,9,...,1425,1426,1427,1428,1429,1430,1431,1432,1433,1434
0,31336,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,Neural_Networks
1,1061127,0,0,0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,Rule_Learning
2,1106406,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Reinforcement_Learning
3,13195,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Reinforcement_Learning
4,37879,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,Probabilistic_Methods


In [ ]:
paper_ids = cora_content.iloc[:, 0]
features = cora_content.iloc[:, 1:-1]
labels = cora_content.iloc[:, -1]

print("Features shape:", features.shape)
print("Labels shape:", labels.shape)


Features shape: (2708, 1433)
Labels shape: (2708,)


In [ ]:
cites_path = "/content/sample_data/cora.cites"


"""
Load the cora.cites file.

Each row represents a citation relationship:
    column 0 → cited paper ID
    column 1 → citing paper ID

The file has no header and is tab-separated.
"""

cora_cites = pd.read_csv(
    cites_path,
    sep="\t",
    header=None
)

cora_cites.columns = ["cited", "citing"]
cora_cites.head()


,cited,citing
0,35,1033
1,35,103482
2,35,103515
3,35,1050679
4,35,1103960


In [ ]:
# Create mapping
"""
Create a mapping from original paper IDs to integer indices.

paper_ids → list of all paper IDs from cora.content
Example:
    31336 → 0
    1061127 → 1
    ...
"""


id_map = {id_: i for i, id_ in enumerate(paper_ids)}




# Map citation IDs

"""
Replace original paper IDs in the citation dataframe
with their mapped integer indices.
"""


cora_cites["cited"] = cora_cites["cited"].map(id_map)
cora_cites["citing"] = cora_cites["citing"].map(id_map)

cora_cites.head()

"""
Neural networks and graph libraries work efficiently with:

-> 0-based indexing

-> Continuous node numbering
"""


'\nNeural networks and graph libraries work efficiently with:\n\n-> 0-based indexing\n\n-> Continuous node numbering\n'

In [ ]:
#Creating a directed graph

G = nx.DiGraph()

# citing → cited
G.add_edges_from(
    zip(cora_cites["citing"], cora_cites["cited"])
)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())


Nodes: 2708
Edges: 5429


In [ ]:
#converting it to undirected for link prediction
G = G.to_undirected()


In [ ]:
"""
Extract all existing edges.
These are positive samples (label = 1).
"""
edges = list(G.edges())
"""
Extract all non-existing edges.
These are potential negative samples (label = 0).
"""
non_edges = list(nx.non_edges(G))

"""
Randomly sample negative edges equal to the number of positive edges.
This ensures class balance.
"""
# Sample negative edges equal to positive edges
np.random.seed(42)
non_edges_sample = np.random.choice(len(non_edges), len(edges), replace=False)
non_edges_sample = [non_edges[i] for i in non_edges_sample]

# Split positive edges
train_edges, test_edges = train_test_split(edges, test_size=0.2, random_state=42)

# Split negative edges
train_non_edges, test_non_edges = train_test_split(non_edges_sample, test_size=0.2, random_state=42)


In [ ]:
node2vec = Node2Vec(
    G,
    dimensions=128,
    walk_length=30,
    num_walks=200,
    workers=4
)

model = node2vec.fit(
    window=10,
    min_count=1,
    batch_words=4
)


In [ ]:
embeddings = {str(node): model.wv[str(node)] for node in G.nodes()}


In [ ]:
def edge_embedding(edge):
    node1, node2 = edge
    emb1 = embeddings[str(node1)]
    emb2 = embeddings[str(node2)]
    return emb1 * emb2


In [ ]:

X_train = []
y_train = []

for edge in train_edges:
    X_train.append(edge_embedding(edge))
    y_train.append(1)

for edge in train_non_edges:
    X_train.append(edge_embedding(edge))
    y_train.append(0)

X_train = np.array(X_train)
y_train = np.array(y_train)

#test split

X_test = []
y_test = []

for edge in test_edges:
    X_test.append(edge_embedding(edge))
    y_test.append(1)

for edge in test_non_edges:
    X_test.append(edge_embedding(edge))
    y_test.append(0)

X_test = np.array(X_test)
y_test = np.array(y_test)


In [ ]:

# Train classifier
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

# Probabilities for AUC & AP
y_pred_proba = clf.predict_proba(X_test)[:, 1]

# Binary predictions (threshold = 0.5)
y_pred = (y_pred_proba >= 0.5).astype(int)

# -------------------------
# Metrics
# -------------------------

# ROC-AUC
auc = roc_auc_score(y_test, y_pred_proba)

# Average Precision
ap = average_precision_score(y_test, y_pred_proba)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("Node2Vec Link Prediction")
print("ROC-AUC:", auc)
print("Average Precision (AP):", ap)

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Node2Vec Link Prediction
ROC-AUC: 0.995671379993113
Average Precision (AP): 0.9950457001434804

Confusion Matrix:
[[1030   26]
 [  26 1030]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98      1056
           1       0.98      0.98      0.98      1056

    accuracy                           0.98      2112
   macro avg       0.98      0.98      0.98      2112
weighted avg       0.98      0.98      0.98      2112



**99% That is almost certainly data leakage. Lets train only on the train edges instead of the whole graph**

In [ ]:
edges = list(G.edges())
train_edges, test_edges = train_test_split(edges, test_size=0.2, random_state=42)

G_train = G.copy()
G_train.remove_edges_from(test_edges)


**p = 1, q = 0.5**

In [ ]:
# p = 1
# q = 0.5


node2vec = Node2Vec(
    G_train,
    dimensions=128,
    walk_length=30,
    num_walks=150,
    p=1,  # return parameter -> probability of going back to the previous node p = 1 means neutral p>1 is means discourage going back p <1 means encourage going back
    q=0.5,  #in-out parameter q < 1  DFS behaviour and q > 1 is BFS behaviour
    workers=1
)

model4 = node2vec.fit(
    window=10,
    min_count=1,
    epochs=5
)


In [ ]:
embeddings = {str(node): model4.wv[str(node)] for node in G_train.nodes()}


non_edges = list(nx.non_edges(G_train))

# Sample equal number of negatives as positives
random.seed(42)
train_non_edges = random.sample(non_edges, len(train_edges))
test_non_edges = random.sample(
    list(set(non_edges) - set(train_non_edges)),
    len(test_edges)
)


In [ ]:
def edge_embedding(edge):
    u, v = edge
    emb_u = embeddings[str(u)]
    emb_v = embeddings[str(v)]
    return emb_u * emb_v  # Hadamard product




In [ ]:
X_train = []
y_train = []

# Positive edges
for edge in train_edges:
    X_train.append(edge_embedding(edge))
    y_train.append(1)

# Negative edges
for edge in train_non_edges:
    X_train.append(edge_embedding(edge))
    y_train.append(0)

X_train = np.array(X_train)
y_train = np.array(y_train)

X_test = []
y_test = []

# Positive test edges (UNSEEN during embedding training)
for edge in test_edges:
    X_test.append(edge_embedding(edge))
    y_test.append(1)

# Negative test edges
for edge in test_non_edges:
    X_test.append(edge_embedding(edge))
    y_test.append(0)

X_test = np.array(X_test)
y_test = np.array(y_test)

In [ ]:


# Train classifier
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

# Probabilities for AUC & AP
y_pred_proba = clf.predict_proba(X_test)[:, 1]

# Binary predictions (threshold = 0.5)
y_pred = (y_pred_proba >= 0.5).astype(int)

# -------------------------
# Metrics
# -------------------------

# ROC-AUC
auc = roc_auc_score(y_test, y_pred_proba)

# Average Precision
ap = average_precision_score(y_test, y_pred_proba)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("Node2Vec Link Prediction")
print("ROC-AUC:", auc)
print("Average Precision (AP):", ap)

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Node2Vec Link Prediction
ROC-AUC: 0.8468576926939854
Average Precision (AP): 0.8804835684095422

Confusion Matrix:
[[1030   26]
 [ 491  565]]

Classification Report:
              precision    recall  f1-score   support

           0       0.68      0.98      0.80      1056
           1       0.96      0.54      0.69      1056

    accuracy                           0.76      2112
   macro avg       0.82      0.76      0.74      2112
weighted avg       0.82      0.76      0.74      2112



**p = 1, q = 0.25**

In [ ]:
# p = 1
# q = 0.25


node2vec = Node2Vec(
    G_train,
    dimensions=128,
    walk_length=30,
    num_walks=150,
    p=1,
    q=0.25,
    workers=1
)

model2 = node2vec.fit(
    window=10,
    min_count=1,
    epochs=5
)


In [ ]:
embeddings = {str(node): model2.wv[str(node)] for node in G_train.nodes()}


In [ ]:
X_train = []
y_train = []

# Positive edges
for edge in train_edges:
    X_train.append(edge_embedding(edge))
    y_train.append(1)

# Negative edges
for edge in train_non_edges:
    X_train.append(edge_embedding(edge))
    y_train.append(0)

X_train = np.array(X_train)
y_train = np.array(y_train)

X_test = []
y_test = []

# Positive test edges (UNSEEN during embedding training)
for edge in test_edges:
    X_test.append(edge_embedding(edge))
    y_test.append(1)

# Negative test edges
for edge in test_non_edges:
    X_test.append(edge_embedding(edge))
    y_test.append(0)

X_test = np.array(X_test)
y_test = np.array(y_test)

In [ ]:
# Train classifier
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

# Probabilities for AUC & AP
y_pred_proba = clf.predict_proba(X_test)[:, 1]

# Binary predictions (threshold = 0.5)
y_pred = (y_pred_proba >= 0.5).astype(int)

# -------------------------
# Metrics
# -------------------------

# ROC-AUC
auc = roc_auc_score(y_test, y_pred_proba)

# Average Precision
ap = average_precision_score(y_test, y_pred_proba)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("Node2Vec Link Prediction")
print("ROC-AUC:", auc)
print("Average Precision (AP):", ap)

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Node2Vec Link Prediction
ROC-AUC: 0.8464622252353077
Average Precision (AP): 0.8774077055033904

Confusion Matrix:
[[1029   27]
 [ 468  588]]

Classification Report:
              precision    recall  f1-score   support

           0       0.69      0.97      0.81      1056
           1       0.96      0.56      0.70      1056

    accuracy                           0.77      2112
   macro avg       0.82      0.77      0.75      2112
weighted avg       0.82      0.77      0.75      2112



**p = 1, q= 0.2**

In [ ]:
# p = 1 , q = 0.2
node2vec = Node2Vec(
    G_train,




    dimensions=128,
    walk_length=30,
    num_walks=150,
    p=1,
    q=0.2,
    workers=1
)

model5 = node2vec.fit(
    window=10,
    min_count=1,
    epochs=5
)

In [ ]:
embeddings = {str(node): model5.wv[str(node)] for node in G_train.nodes()}
# training dta

X_train = []
y_train = []

# Positive edges
for edge in train_edges:
    X_train.append(edge_embedding(edge))
    y_train.append(1)

# Negative edges
for edge in train_non_edges:
    X_train.append(edge_embedding(edge))
    y_train.append(0)

X_train = np.array(X_train)
y_train = np.array(y_train)

#testing data
X_test = []
y_test = []

# Positive test edges (UNSEEN during embedding training)
for edge in test_edges:
    X_test.append(edge_embedding(edge))
    y_test.append(1)

# Negative test edges
for edge in test_non_edges:
    X_test.append(edge_embedding(edge))
    y_test.append(0)

X_test = np.array(X_test)
y_test = np.array(y_test)

# Train classifier
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

# Probabilities for AUC & AP
y_pred_proba = clf.predict_proba(X_test)[:, 1]

# Binary predictions (threshold = 0.5)
y_pred = (y_pred_proba >= 0.5).astype(int)

# -------------------------
# Metrics
# -------------------------

# ROC-AUC
auc = roc_auc_score(y_test, y_pred_proba)

# Average Precision
ap = average_precision_score(y_test, y_pred_proba)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("Node2Vec Link Prediction")
print("p = 1 and q = 0.2")
print("ROC-AUC:", auc)
print("Average Precision (AP):", ap)

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Node2Vec Link Prediction
p = 1 and q = 0.2
ROC-AUC: 0.8628118005337465
Average Precision (AP): 0.8871245086948416

Confusion Matrix:
[[1030   26]
 [ 466  590]]

Classification Report:
              precision    recall  f1-score   support

           0       0.69      0.98      0.81      1056
           1       0.96      0.56      0.71      1056

    accuracy                           0.77      2112
   macro avg       0.82      0.77      0.76      2112
weighted avg       0.82      0.77      0.76      2112



**p = 1, q = 0.15**

In [ ]:
# p = 1 , q = 0.15
node2vec = Node2Vec(
    G_train,
    dimensions=128,
    walk_length=30,
    num_walks=150,
    p=1,
    q=0.15,
    workers=1
)

model6 = node2vec.fit(
    window=10,
    min_count=1,
    epochs=5
)

In [ ]:
embeddings = {str(node): model6.wv[str(node)] for node in G_train.nodes()}
# training dta

X_train = []
y_train = []

# Positive edges
for edge in train_edges:
    X_train.append(edge_embedding(edge))
    y_train.append(1)

# Negative edges
for edge in train_non_edges:
    X_train.append(edge_embedding(edge))
    y_train.append(0)

X_train = np.array(X_train)
y_train = np.array(y_train)

#testing data
X_test = []
y_test = []

# Positive test edges (UNSEEN during embedding training)
for edge in test_edges:
    X_test.append(edge_embedding(edge))
    y_test.append(1)

# Negative test edges
for edge in test_non_edges:
    X_test.append(edge_embedding(edge))
    y_test.append(0)

X_test = np.array(X_test)
y_test = np.array(y_test)

# Train classifier
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

# Probabilities for AUC & AP
y_pred_proba = clf.predict_proba(X_test)[:, 1]

# Binary predictions (threshold = 0.5)
y_pred = (y_pred_proba >= 0.5).astype(int)

# -------------------------
# Metrics
# -------------------------

# ROC-AUC
auc = roc_auc_score(y_test, y_pred_proba)

# Average Precision
ap = average_precision_score(y_test, y_pred_proba)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("Node2Vec Link Prediction")
print("p = 1 and q = 0.15")
print("ROC-AUC:", auc)
print("Average Precision (AP):", ap)

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Node2Vec Link Prediction
p = 1 and q = 0.15
ROC-AUC: 0.8544092379763545
Average Precision (AP): 0.8844040740613784

Confusion Matrix:
[[1030   26]
 [ 476  580]]

Classification Report:
              precision    recall  f1-score   support

           0       0.68      0.98      0.80      1056
           1       0.96      0.55      0.70      1056

    accuracy                           0.76      2112
   macro avg       0.82      0.76      0.75      2112
weighted avg       0.82      0.76      0.75      2112



**PubMed Dataset**

In [ ]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 13.1 MB/s eta 0:00:00


In [ ]:
import torch
import networkx as nx
from torch_geometric.datasets import Planetoid
from torch_geometric.utils import to_networkx

# Load PubMed dataset
dataset = Planetoid(root='/tmp/PubMed', name='PubMed')
data = dataset[0]

print(data)

Data(x=[19717, 500], edge_index=[2, 88648], y=[19717], train_mask=[19717], val_mask=[19717], test_mask=[19717])


In [ ]:
# Convert to NetworkX graph
G = to_networkx(data, to_undirected=True)

print("Number of nodes:", G.number_of_nodes())
print("Number of edges:", G.number_of_edges())

Number of nodes: 19717
Number of edges: 44324


In [ ]:
# Relabel nodes as strings (important for gensim)
G = nx.relabel_nodes(G, lambda x: str(x))

In [ ]:
edges = list(G.edges())
train_edges, test_edges = train_test_split(edges, test_size=0.2, random_state=42)

G_train = G.copy()
G_train.remove_edges_from(test_edges)
# p = 1
# q = 0.5


node2vec = Node2Vec(
    G_train,
    dimensions=128,
    walk_length=30,
    num_walks=150,
    p=1,  # return parameter -> probability of going back to the previous node p = 1 means neutral p>1 is means discourage going back p <1 means encourage going back
    q=0.5,  #in-out parameter q < 1  DFS behaviour and q > 1 is BFS behaviour
    workers=1
)

model4 = node2vec.fit(
    window=10,
    min_count=1,
    epochs=5
)
embeddings = {str(node): model4.wv[str(node)] for node in G_train.nodes()}


non_edges = list(nx.non_edges(G_train))

# Sample equal number of negatives as positives
random.seed(42)
train_non_edges = random.sample(non_edges, len(train_edges))
test_non_edges = random.sample(
    list(set(non_edges) - set(train_non_edges)),
    len(test_edges)
)
def edge_embedding(edge):
    u, v = edge
    emb_u = embeddings[str(u)]
    emb_v = embeddings[str(v)]
    return emb_u * emb_v  # Hadamard product
X_train = []
y_train = []

# Positive edges
for edge in train_edges:
    X_train.append(edge_embedding(edge))
    y_train.append(1)

# Negative edges
for edge in train_non_edges:
    X_train.append(edge_embedding(edge))
    y_train.append(0)

X_train = np.array(X_train)
y_train = np.array(y_train)

X_test = []
y_test = []

# Positive test edges (UNSEEN during embedding training)
for edge in test_edges:
    X_test.append(edge_embedding(edge))
    y_test.append(1)

# Negative test edges
for edge in test_non_edges:
    X_test.append(edge_embedding(edge))
    y_test.append(0)

X_test = np.array(X_test)
y_test = np.array(y_test)



# Train classifier
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

# Probabilities for AUC & AP
y_pred_proba = clf.predict_proba(X_test)[:, 1]

# Binary predictions (threshold = 0.5)
y_pred = (y_pred_proba >= 0.5).astype(int)

# -------------------------
# Metrics
# -------------------------

# ROC-AUC
auc = roc_auc_score(y_test, y_pred_proba)

# Average Precision
ap = average_precision_score(y_test, y_pred_proba)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("Node2Vec Link Prediction")
print("ROC-AUC:", auc)
print("Average Precision (AP):", ap)

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))







**MATRIX FACTORIZATION METHOD OF LINK PREDICTION**

**Spectral / SVD-based MF**

In [ ]:

# Ensure integer labels
G = nx.convert_node_labels_to_integers(G)
G = G.to_undirected()

edges = list(G.edges())
train_edges, test_edges = train_test_split(edges, test_size=0.2, random_state=42)

G_train = G.copy()
G_train.remove_edges_from(test_edges)


A = nx.to_numpy_array(G_train)

A2 = A @ A
M = A + 0.5 * A2  # 2-hop proximity

# Normalize
degree = np.sum(M, axis=1)
D_inv_sqrt = np.diag(1.0 / np.sqrt(degree + 1e-10))
M_norm = D_inv_sqrt @ M @ D_inv_sqrt

In [ ]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=256, random_state=42)
node_embeddings = svd.fit_transform(M_norm)

print("Embedding shape:", node_embeddings.shape)


Embedding shape: (2708, 256)


In [ ]:
non_edges = list(nx.non_edges(G_train))
random.seed(42)


train_non_edges = random.sample(non_edges, len(train_edges))
remaining = list(set(non_edges) - set(train_non_edges))
test_non_edges = random.sample(remaining, len(test_edges))



In [ ]:
def edge_embedding(edge):
    u, v = edge
    emb_u = node_embeddings[u]
    emb_v = node_embeddings[v]

    return np.concatenate([
        emb_u * emb_v,          # Hadamard
        np.abs(emb_u - emb_v)   # L1 distance
    ])


In [ ]:
X_train = []
y_train = []

for edge in train_edges:
    X_train.append(edge_embedding(edge))
    y_train.append(1)

for edge in train_non_edges:
    X_train.append(edge_embedding(edge))
    y_train.append(0)

X_train = np.array(X_train)

y_train = np.array(y_train)

X_test = []
y_test = []

for edge in test_edges:
    X_test.append(edge_embedding(edge))
    y_test.append(1)

for edge in test_non_edges:
    X_test.append(edge_embedding(edge))
    y_test.append(0)

X_test = np.array(X_test)
y_test = np.array(y_test)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

y_pred_proba = clf.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_pred_proba)
ap = average_precision_score(y_test, y_pred_proba)


print("Matrix Factorization AUC:", auc)
print("Matrix Factorization AP:", ap)


Matrix Factorization AUC: 0.7631158890036731
Matrix Factorization AP: 0.7810579425050512


**Pure Adjacency Matrix Factorization (SVD on A)**

In [ ]:
import numpy as np
import networkx as nx
from sklearn.decomposition import TruncatedSVD

# -----------------------------------------
# Step 1: Convert graph to adjacency matrix
# -----------------------------------------
A = nx.to_numpy_array(G_train)

"""
A is the adjacency matrix of size (N x N)

A[i, j] = 1 if edge exists
A[i, j] = 0 otherwise
"""

# -----------------------------------------
# Step 2: Low-rank factorization using SVD
# -----------------------------------------
svd = TruncatedSVD(n_components=128, random_state=42)

"""
TruncatedSVD computes:

A ≈ U_k Σ_k V_k^T

We use:
Z = U_k Σ_k
"""

node_embeddings = svd.fit_transform(A)

print("Embedding shape:", node_embeddings.shape)

Embedding shape: (2708, 128)


In [ ]:
non_edges = list(nx.non_edges(G_train))
random.seed(42)


train_non_edges = random.sample(non_edges, len(train_edges))
remaining = list(set(non_edges) - set(train_non_edges))
test_non_edges = random.sample(remaining, len(test_edges))




In [ ]:
X_train = []
y_train = []

for edge in train_edges:
    X_train.append(edge_embedding(edge))
    y_train.append(1)

for edge in train_non_edges:
    X_train.append(edge_embedding(edge))
    y_train.append(0)

X_train = np.array(X_train)

y_train = np.array(y_train)

X_test = []
y_test = []

for edge in test_edges:
    X_test.append(edge_embedding(edge))
    y_test.append(1)

for edge in test_non_edges:
    X_test.append(edge_embedding(edge))
    y_test.append(0)

X_test = np.array(X_test)
y_test = np.array(y_test)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

y_pred_proba = clf.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_pred_proba)
ap = average_precision_score(y_test, y_pred_proba)


print("Matrix Factorization AUC:", auc)
print("Matrix Factorization AP:", ap)


Matrix Factorization AUC: 0.5790647956841138
Matrix Factorization AP: 0.5967139169160531


**Higher-Order Proximity MF (A + A² + A³)**

In [ ]:
# -----------------------------------------
# Compute higher-order adjacency matrices
# -----------------------------------------

A2 = A @ A      # 2-hop connectivity
A3 = A2 @ A     # 3-hop connectivity

"""
A2[i, j] = number of 2-hop paths between i and j
A3[i, j] = number of 3-hop paths
"""

# Weighted combination
M = A + 0.5*A2 + 0.25*A3

# -----------------------------------------
# Normalize matrix (optional but helpful)
# -----------------------------------------

degree = np.sum(M, axis=1)
D_inv_sqrt = np.diag(1.0 / np.sqrt(degree + 1e-10))

"""
Symmetric normalization:

M_norm = D^{-1/2} M D^{-1/2}

Prevents high-degree nodes from dominating.
"""

M_norm = D_inv_sqrt @ M @ D_inv_sqrt

# -----------------------------------------
# Factorization
# -----------------------------------------

svd = TruncatedSVD(n_components=128, random_state=42)
node_embeddings = svd.fit_transform(M_norm)

print("Embedding shape:", node_embeddings.shape)

Embedding shape: (2708, 128)


In [ ]:
non_edges = list(nx.non_edges(G_train))
random.seed(42)


train_non_edges = random.sample(non_edges, len(train_edges))
remaining = list(set(non_edges) - set(train_non_edges))
test_non_edges = random.sample(remaining, len(test_edges))
X_train = []
y_train = []

for edge in train_edges:
    X_train.append(edge_embedding(edge))
    y_train.append(1)

for edge in train_non_edges:
    X_train.append(edge_embedding(edge))
    y_train.append(0)

X_train = np.array(X_train)

y_train = np.array(y_train)

X_test = []
y_test = []

for edge in test_edges:
    X_test.append(edge_embedding(edge))
    y_test.append(1)

for edge in test_non_edges:
    X_test.append(edge_embedding(edge))
    y_test.append(0)

X_test = np.array(X_test)
y_test = np.array(y_test)

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

y_pred_proba = clf.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_pred_proba)
ap = average_precision_score(y_test, y_pred_proba)


print("Matrix Factorization AUC:", auc)
print("Matrix Factorization AP:", ap)


Matrix Factorization AUC: 0.5464266241965106
Matrix Factorization AP: 0.5321967508676796


**Normalized Laplacian Factorization**

In [ ]:
# -----------------------------------------
# Degree normalization
# -----------------------------------------

degree = np.sum(A, axis=1)
D_inv_sqrt = np.diag(1.0 / np.sqrt(degree + 1e-10))

"""
Normalized adjacency matrix:

L = D^{-1/2} A D^{-1/2}
"""

L = D_inv_sqrt @ A @ D_inv_sqrt

# -----------------------------------------
# Factorization
# -----------------------------------------

svd = TruncatedSVD(n_components=128, random_state=42)
node_embeddings = svd.fit_transform(L)

print("Embedding shape:", node_embeddings.shape)

Embedding shape: (2708, 128)


In [ ]:
non_edges = list(nx.non_edges(G_train))
random.seed(42)


train_non_edges = random.sample(non_edges, len(train_edges))
remaining = list(set(non_edges) - set(train_non_edges))
test_non_edges = random.sample(remaining, len(test_edges))
X_train = []
y_train = []

for edge in train_edges:
    X_train.append(edge_embedding(edge))
    y_train.append(1)

for edge in train_non_edges:
    X_train.append(edge_embedding(edge))
    y_train.append(0)

X_train = np.array(X_train)

y_train = np.array(y_train)

X_test = []
y_test = []

for edge in test_edges:
    X_test.append(edge_embedding(edge))
    y_test.append(1)

for edge in test_non_edges:
    X_test.append(edge_embedding(edge))
    y_test.append(0)

X_test = np.array(X_test)
y_test = np.array(y_test)

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

y_pred_proba = clf.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_pred_proba)
ap = average_precision_score(y_test, y_pred_proba)


print("Matrix Factorization AUC:", auc)
print("Matrix Factorization AP:", ap)


Matrix Factorization AUC: 0.5593048740243343
Matrix Factorization AP: 0.5612132018949776


**Non-Negative Matrix Factorization (NMF)**

In [ ]:
from sklearn.decomposition import NMF

"""
NMF minimizes:

||A - WH||_F^2
Subject to W, H ≥ 0
"""

nmf = NMF(n_components=128, init='random', random_state=42, max_iter=500)

node_embeddings = nmf.fit_transform(A)

print("Embedding shape:", node_embeddings.shape)

Embedding shape: (2708, 128)


In [ ]:
non_edges = list(nx.non_edges(G_train))
random.seed(42)


train_non_edges = random.sample(non_edges, len(train_edges))
remaining = list(set(non_edges) - set(train_non_edges))
test_non_edges = random.sample(remaining, len(test_edges))
X_train = []
y_train = []

for edge in train_edges:
    X_train.append(edge_embedding(edge))
    y_train.append(1)

for edge in train_non_edges:
    X_train.append(edge_embedding(edge))
    y_train.append(0)

X_train = np.array(X_train)

y_train = np.array(y_train)

X_test = []
y_test = []

for edge in test_edges:
    X_test.append(edge_embedding(edge))
    y_test.append(1)

for edge in test_non_edges:
    X_test.append(edge_embedding(edge))
    y_test.append(0)

X_test = np.array(X_test)
y_test = np.array(y_test)

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

y_pred_proba = clf.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_pred_proba)
ap = average_precision_score(y_test, y_pred_proba)


print("Matrix Factorization AUC:", auc)
print("Matrix Factorization AP:", ap)


Matrix Factorization AUC: 0.5904521959653352
Matrix Factorization AP: 0.6108316229743815


**PMI-based Matrix Factorization (DeepWalk Approximation)**

In [ ]:
# -----------------------------------------
# Compute A^2 for co-occurrence
# -----------------------------------------

A2 = A @ A

# Graph volume
vol = np.sum(A)

degree = np.sum(A, axis=1)

"""
PMI matrix computation:

PMI[i,j] = log( (vol * A2[i,j]) / (deg[i]*deg[j]) )
"""

PMI = np.zeros_like(A2)

for i in range(A.shape[0]):
    for j in range(A.shape[1]):
        if A2[i, j] > 0:
            PMI[i, j] = np.log(
                (vol * A2[i, j]) / (degree[i] * degree[j] + 1e-10)
            )

# Replace negative values with 0 (PPMI)
PMI = np.maximum(PMI, 0)

# -----------------------------------------
# Factorization
# -----------------------------------------

svd = TruncatedSVD(n_components=128, random_state=42)
node_embeddings = svd.fit_transform(PMI)

print("Embedding shape:", node_embeddings.shape)

Embedding shape: (2708, 128)


In [ ]:
non_edges = list(nx.non_edges(G_train))
random.seed(42)


train_non_edges = random.sample(non_edges, len(train_edges))
remaining = list(set(non_edges) - set(train_non_edges))
test_non_edges = random.sample(remaining, len(test_edges))
X_train = []
y_train = []

for edge in train_edges:
    X_train.append(edge_embedding(edge))
    y_train.append(1)

for edge in train_non_edges:
    X_train.append(edge_embedding(edge))
    y_train.append(0)

X_train = np.array(X_train)

y_train = np.array(y_train)

X_test = []
y_test = []

for edge in test_edges:
    X_test.append(edge_embedding(edge))
    y_test.append(1)

for edge in test_non_edges:
    X_test.append(edge_embedding(edge))
    y_test.append(0)

X_test = np.array(X_test)
y_test = np.array(y_test)

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

y_pred_proba = clf.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, y_pred_proba)
ap = average_precision_score(y_test, y_pred_proba)


print("Matrix Factorization AUC:", auc)
print("Matrix Factorization AP:", ap)


Matrix Factorization AUC: 0.5578548266758494
Matrix Factorization AP: 0.5704043162789072


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import networkx as nx

In [ ]:


# Convert training graph to adjacency matrix
A = nx.to_numpy_array(G_train)

# Convert to torch tensor
A = torch.tensor(A, dtype=torch.float32)

N = A.shape[0]   # number of nodes
embedding_dim = 128

In [ ]:
class MatrixFactorization(nn.Module):
    """
    Implements:

    A ≈ Z Z^T

    Z is learnable embedding matrix.
    """

    def __init__(self, num_nodes, dim):
        super().__init__()

        # Z initialized randomly
        self.Z = nn.Parameter(torch.randn(num_nodes, dim))

    def forward(self):
        """
        Reconstruct adjacency matrix:

        A_hat = Z Z^T
        """
        return self.Z @ self.Z.t()

In [ ]:
model = MatrixFactorization(N, embedding_dim)

optimizer = optim.Adam(model.parameters(), lr=0.01)

lambda_reg = 1e-4   # regularization strength
epochs = 300

In [ ]:
for epoch in range(epochs):

    optimizer.zero_grad()

    A_hat = model()

    # Reconstruction loss
    recon_loss = torch.norm(A - A_hat, p='fro')**2

    # Regularization
    reg_loss = lambda_reg * torch.norm(model.Z, p='fro')**2

    loss = recon_loss + reg_loss

    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 976983616.0000
Epoch 50, Loss: 223880512.0000
Epoch 100, Loss: 84307928.0000
Epoch 150, Loss: 41637272.0000
Epoch 200, Loss: 23773696.0000
Epoch 250, Loss: 14895725.0000


In [ ]:
node_embeddings = model.Z.detach().numpy()

In [ ]:
def score(u, v):
    return np.dot(node_embeddings[u], node_embeddings[v])

In [ ]:
from sklearn.metrics import roc_auc_score, average_precision_score

y_true = []
y_scores = []

# Positive edges
for u, v in test_edges:
    y_true.append(1)
    y_scores.append(score(u, v))

# Negative edges
for u, v in test_non_edges:
    y_true.append(0)
    y_scores.append(score(u, v))

auc = roc_auc_score(y_true, y_scores)
ap = average_precision_score(y_true, y_scores)

print("Regularized MF AUC:", auc)
print("Regularized MF AP:", ap)

Regularized MF AUC: 0.48548203985881544
Regularized MF AP: 0.48109458868409155
